# Advanced Pipeline Configuration Guide
# Pipeline 高级配置指南

This tutorial covers advanced features of ModelPipeline:
本教程介绍 ModelPipeline 的高级功能：

1. **PipelineConfigs**: Customize initialization, training, and prediction parameters for each model
1. **PipelineConfigs**: 自定义每个模型的初始化、训练和预测参数

2. **Model filtering**: include_models / exclude_models
2. **模型筛选**: include_models / exclude_models

3. **Custom Scaler**: Use different data scalers
3. **自定义 Scaler**: 使用不同的数据缩放器

4. **Custom evaluation metric**: Replace the default MAE
4. **自定义评估指标**: 替换默认的 MAE

5. **Double-underscore syntax**: Pass model parameters via `model_name__param=value`
5. **双下划线语法**: 通过 `model_name__param=value` 传递模型参数

6. **Model query and configuration retrieval**
6. **模型查询与配置获取**

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 200
dates = pd.date_range(start='2020-01-01', periods=n, freq='D')
values = np.sin(np.linspace(0, 4 * np.pi, n)) + np.random.randn(n) * 0.1
data = pd.DataFrame({'date': dates, 'value': values})

LAGS = 12

## 1. PipelineConfigs: Custom Configuration
## 1. PipelineConfigs 自定义配置

PipelineConfigs allows you to create multiple variants of the same model, each with different hyperparameters.
PipelineConfigs 允许你为同一个模型创建多个变体，每个变体使用不同的超参数。

In [ ]:
from PipelineTS.pipeline import ModelPipeline, PipelineConfigs

# Create configs: two LightGBM variants
# 创建配置：两个 LightGBM 变体
configs = PipelineConfigs([
    ('lightgbm', 'lgbm_small', {
        'init_configs': {'n_estimators': 50},
        'fit_configs': {}
    }),
    ('lightgbm', 'lgbm_large', {
        'init_configs': {'n_estimators': 300},
        'fit_configs': {}
    }),
])

In [ ]:
pipeline = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['lightgbm'],
    configs=configs,
    quantile=None, cv=3
)

leaderboard = pipeline.fit(data)
leaderboard

## 2. Model Filtering
## 2. 模型筛选

Control which models are trained via `include_models` and `exclude_models`.
通过 `include_models` 和 `exclude_models` 控制训练哪些模型。

In [ ]:
# List all available models / 查看所有可用模型
print("All available models / 所有可用模型:")
print(ModelPipeline.list_all_available_models())

# Predefined model sets / 预定义模型集合:
# 'light' - Lightweight models (default, speed priority) / 轻量级模型（默认，速度优先）
# 'all'   - All models / 所有模型
# 'nn'    - Neural network models only / 仅神经网络模型
# 'ml'    - Machine learning models only / 仅机器学习模型

In [ ]:
# Use only ML models / 仅使用 ML 模型
pipeline_ml = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models='ml', quantile=None, cv=2
)
leaderboard_ml = pipeline_ml.fit(data)
print("ML model leaderboard / ML 模型排行榜:")
leaderboard_ml

In [ ]:
# Specify a custom model list / 指定具体模型列表
pipeline_custom = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['lightgbm', 'xgboost', 'd_linear', 'n_linear'],
    quantile=None, cv=2
)
leaderboard_custom = pipeline_custom.fit(data)
leaderboard_custom

## 3. Custom Scaler
## 3. 自定义 Scaler

By default, MinMaxScaler is used. You can replace it with any sklearn TransformerMixin-compatible scaler.
默认使用 MinMaxScaler，你可以替换为任何 sklearn TransformerMixin 兼容的缩放器。

In [ ]:
from sklearn.preprocessing import StandardScaler
from PipelineTS.preprocessing import Scaler

# Method 1: Use sklearn's StandardScaler / 方法 1：使用 sklearn 的 StandardScaler
pipeline_std = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['lightgbm'],
    scaler=StandardScaler(),
    quantile=None, cv=2
)

# Method 2: Use PipelineTS built-in Scaler / 方法 2：使用 PipelineTS 内置的 Scaler
# Supported: 'min_max', 'standard', 'quantile', 'gauss_rank'
# 支持：'min_max', 'standard', 'quantile', 'gauss_rank'
scaler = Scaler('gauss_rank')

# Method 3: Disable data scaling / 方法 3：关闭数据缩放
pipeline_no_scale = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['lightgbm'],
    scaler=None,   # No scaler / 不使用缩放器
    quantile=None, cv=2
)

## 4. Custom Evaluation Metric
## 4. 自定义评估指标

In [ ]:
from PipelineTS.spinesTS.metrics import rmse, wmape

# Use RMSE as the evaluation metric / 使用 RMSE 作为评估指标
pipeline_rmse = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['lightgbm', 'xgboost'],
    metric=rmse,                # Custom metric function / 自定义指标函数
    metric_less_is_better=True, # Lower RMSE is better / RMSE 越小越好
    quantile=None, cv=2
)
leaderboard_rmse = pipeline_rmse.fit(data)
leaderboard_rmse

## 5. Double-underscore Syntax for Model Parameters
## 5. 通过双下划线语法传递模型参数

Use `model_name__param_name=value` format to pass model initialization parameters.
使用 `模型名__参数名=值` 的格式传递模型初始化参数。

In [ ]:
pipeline_kwargs = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['lightgbm', 'xgboost'],
    quantile=None, cv=2,
    lightgbm__n_estimators=100,    # LightGBM specific / LightGBM 专属参数
    xgboost__n_estimators=150,     # XGBoost specific / XGBoost 专属参数
    xgboost__verbose=0
)
leaderboard_kwargs = pipeline_kwargs.fit(data)
leaderboard_kwargs

## 6. Model Query and Configuration Retrieval
## 6. 模型查询与配置获取

In [ ]:
# Get the best model / 获取最佳模型
best_model = pipeline_kwargs.get_model()
print(f"Best model / 最佳模型: {type(best_model).__name__}")

# Get a specific model / 获取指定模型
model_name = pipeline_kwargs.leader_board_.iloc[0]['model']
specific_model = pipeline_kwargs.get_model(model_name)
print(f"Specified model / 指定模型: {model_name}")

# Get all configurations for a model / 获取模型全部配置
configs = pipeline_kwargs.get_model_all_configs()
print(f"Model configs / 模型配置: {configs}")

In [ ]:
# Predict using a specific model / 使用指定模型进行预测
result = pipeline_kwargs.predict(10, model_name=model_name)
result